In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, sha2, col
spark = SparkSession.builder.enableHiveSupport().getOrCreate()
df = spark.read.parquet("/user/hive/warehouse/sclsepb4532_staging.db/clickstream_logs_stg", header=True, inferSchema=True)

26/11/09 12:30:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 12:30:17 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/11/09 12:32:13 WARN DataStreamer: Slow waitForAckedSeqno took 63195ms (threshold=30000ms). File being written: /user/sclsepb4532/.sparkStaging/application_1793087731018_0449/__spark_libs__7899864821578593676.zip, block: BP-1364978691-172.31.64.10-1677754292516:blk_1101629030_27897082, Write pipeline datanodes: [DatanodeInfoWithStorage[172.31.64.12:9866,DS-c7b06f49-e29e-4fd7-8923-882da8ea1f7c,DISK], DatanodeInfoWithStorage[172.31.73.191:9866,DS-e2b7285d-a9a5-47f8-bcee-e91f66a4d588,DISK], DatanodeInfoWithStorage[172.31.64.11:9866,DS-8af7bd92-c7a4-44fb-b361-00496b8456c3,DISK]].


In [2]:
df = (df.withColumn('customer_key',sha2(col("customer_id"), 256)).drop('customer_id')
       .withColumn('session_key',sha2(col("session_id"), 256)).drop('session_id')
        .withColumn('product_key',sha2(col("product_id"), 256)).drop('product_id'))

In [3]:
df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .saveAsTable("sclsepb4532_analytics.clickstream_logs_analytics")

26/11/09 12:42:18 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
